# `10_error_analysis.ipynb` — Linguistic & Systems Error Analysis (105–120 min CAS Session)

## 1. Where Does QLoRA Beat 10-Shot Prompting?
Aggregate Micro-F1 (`0.8360` for `Q1` QLoRA vs. `0.7052` for `P3` 10-shot on `test_syn`) tells us *that* fine-tuning wins, but not *why*. Below we decompose the gain along four linguistic and systems dimensions:
1. **Rare-Category Recall vs. Support:** In a 10-shot prompt, only ~8–12 categories can appear in the demonstration examples. Categories with subtle boundaries (`Atmosphäre`, `Design`, `Gepäck`, `Barrierefreiheit`, `Auslastung_und_Platzangebot`) gain **+20 to +35 F1 points** from QLoRA.
2. **Irrelevant Document (`relevance=false`) Precision:** Prompted LLMs suffer from *helpfulness bias*—when given a tweet mentioning *"Bahnhof"*, they try hard to assign `Allgemein:neutral` or `Image:negative` even when the tweet is not customer feedback about DB. Fine-tuning teaches the exact boundary of `{"labels": []}`.
3. **Paired Document Contingency Quadrants (`McNemar` / Paired Win Analysis):** On `test_syn` (`2,566` docs), QLoRA gets the **entire document label set** right on **621 documents where `P3` failed**, while `P3` wins on only 113 documents.
4. **Synchronic (`test_syn`) vs. Diachronic (`test_dia`) Drift:** Both prompting and fine-tuning experience a ~1.9–2.1 F1 drop on `test_TIMESTAMP2` (collected 6 months later during winter rail disruptions), showing that QLoRA adapts to the domain taxonomy without overfitting exclusively to summer 2016 topics.

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import Image, display, Markdown
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
    os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from germeval_llm.convert import ensure_derived_datasets_ready
ensure_derived_datasets_ready()
print("Repository initialized:", ROOT.name)


In [ ]:
display(Image(filename='results/figures/error_analysis_delta_f1.png'))

quad = json.loads(Path('results/metrics/error_analysis_quadrants_test_syn.json').read_text())
print('Paired Document Exact-Set Quadrants on test_syn (P3 10-shot vs Q1 QLoRA):')
print(json.dumps({k: v for k, v in quad.items() if k != 'sample_qlora_wins' and k != 'sample_prompt_wins'}, indent=2))
print('\nSample documents where QLoRA succeeded and Prompting failed:')
for ex in quad['sample_qlora_wins'][:3]:
    print('-', ex['doc_id'], '| Gold:', ex['gold'], '| P3:', ex['prompt_pred'], '| QLoRA:', ex['qlora_pred'])

In [ ]:
# Synchronic (test_syn) vs Diachronic (test_dia) generalization comparison
bench = pd.read_csv('results/metrics/benchmark.csv')
piv = bench[bench['run_id'].isin(['P1_guidelines_0shot', 'P3_dev_optimized_10shot', 'L1_lora_r16', 'Q1_qlora_r16'])].pivot(
    index='run_id', columns='split', values='c2_strict_micro_f1'
)[['dev', 'test_syn', 'test_dia']]
piv['diachronic_drop (syn - dia)'] = (piv['test_syn'] - piv['test_dia']).round(4)
display(piv)

### Final CAS Decision Matrix: When Should You Prompt vs. Fine-Tune?
| Criterion | Guideline 0-Shot (`P1`) | 10-Shot ICL (`P3`) | 4-bit NF4 QLoRA (`Q1`) |
|---|---|---|---|
| **Labeled data required** | `0` docs | `10` docs | `$\ge 250\text{–}500$` docs (crossover) |
| **Task C2 Strict Micro-F1 (`test_syn`)** | `0.644` | `0.705` | **`0.836`** (`+13.1` pts vs `P3`) |
| **Task D Explicit-Span F1 (`test_syn`)** | `0.453` (`D1`) | `0.453` (`D1`) | **`0.650`** (`+19.7` pts vs `D1`) |
| **Raw JSON Schema Compliance** | `~60–94%` (fences/drift) | `~98.8%` | **`99.9%+`** (`VALID_EXACT`) |
| **Inference Prompt Tokens / 1k Docs** | `~550k` tokens | `~1,670k` tokens (**3× cost**) | **`~550k` tokens** (lean 0-shot prompt) |
| **Peak Training GPU VRAM** | `0 GB` | `0 GB` | **`~2.9 GB`** (fits any T4 / L4 GPU) |